> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 10 · Notebook 07 — ML strategies: meta-labeled momentum, volatility and crash warnings

**Sessions:** S13–S14 (ML strategy I: direction with meta-labeling · ML strategy II: ranking, regimes, volatility & rare events) · [Lesson plan](../../docs/lessons/PART_10_MACHINE_LEARNING.md) · graded labs in [`labs/part10/`](../../labs/part10/)

**You will:**
1. Run a walk-forward meta-labeled momentum strategy with purged retraining.
2. Forecast volatility with HAR and with gradient boosting.
3. Build a crash warning, judge it by PR-AUC, and use it only to cut risk.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known structure (a hidden regime, a known autocorrelation, planted importances, pure noise), so you can see what a model can learn, and that it learns nothing from noise.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p10lib.py is in notebooks/part10/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p10lib as p

p.use_course_style()
from sklearn.metrics import roc_auc_score

## 1. Meta-labeled momentum, walk-forward

Every 100 events, retrain the meta-model and trade the next block. The model may only learn from events whose label was **already resolved**: their `t1` is strictly before the current event's `t0`. Anything else trains on outcomes that haven't happened yet (a subtle leak that a simple `ds.iloc[:s]` makes).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def walk_forward_meta(ds, kind="rf", start=1 / 3, retrain_every=100, cost_bps=5.0):
    n, cost, rows = len(ds), cost_bps / 1e4, []
    for s in range(int(start * n), n, retrain_every):
        known = ds.iloc[:s][ds["t1"].iloc[:s] < ds.index[s]]
        m = p.fit_weighted(p.make_model(kind), known[p.FEATURES], known["y"], known["w"])
        block = ds.iloc[s:s + retrain_every]
        prob = m.predict_proba(block[p.FEATURES])[:, 1]
        size = p.bet_size(prob)
        rows.append(pd.DataFrame({"prob": prob, "size": size, "y": block["y"], "primary": block["ret"] - 2 * cost,
                                  "meta": size * block["ret"] - 2 * cost * size}, index=block.index))
    return pd.concat(rows)

mk = p.regime_market(6000, seed=1)                       # the labs' market: 24 years, hidden trend/chop regimes
ds = p.meta_dataset(mk)
mine = p.check("walk_forward_meta", p.attempt(walk_forward_meta, ds, "logit"), p.walk_forward_meta(ds, "logit"))
print(f"{len(ds)} events; {len(mine)} traded out of sample, retrained every 100 events")

In [ ]:
oos = {kind: (mine if kind == "logit" else p.walk_forward_meta(ds, kind)) for kind in ("logit", "rf")}
years = (mine.index[-1] - mine.index[0]).days / 365.25
table = pd.concat({kind: p.meta_summary(o, years) for kind, o in oos.items()})
display(table.round(3))
regime = mk["regime"].loc[oos["rf"].index].map({0: "chop", 1: "trend"})
print("random forest: mean bet size by hidden regime", oos["rf"]["size"].groupby(regime).mean().round(3).to_dict(),
      f"| AUC {roc_auc_score(oos['rf']['y'], oos['rf']['prob']):.3f}")

The primary momentum rule loses after costs. The meta-model's AUC is only about 0.54, yet it turns the strategy positive: it bets several times more in the trend regime than in chop, and skips many losing trades. Meta-labeling rarely needs a strong model; it needs one that knows when to stay small. (Two model families tried: log both for the Deflated Sharpe Ratio.)

## 2. Volatility: the simple model wins

HAR (Corsi 2009) forecasts tomorrow's volatility proxy `v_{t+1}` (here `|return|`) from `d = v_t`, `w` = the mean of the last 5 values and `m` = the mean of the last 22, all ending at `t`. Build the table with `target = v.shift(-1)` and drop the NaN rows.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def har_features(absret):
    return pd.DataFrame({"d": absret, "w": absret.rolling(5).mean(), "m": absret.rolling(22).mean(),
                         "target": absret.shift(-1)}).dropna()

g = p.garch_returns()                                    # GARCH volatility: clusters, so it is forecastable
cases = [g["ret"].abs(), g["ret"].abs().iloc[:60]]
mine = [p.attempt(har_features, c) for c in cases]
mine = p.check("har_features", mine, [p.har_features(c) for c in cases])
har = mine[0]
fc = p.vol_forecasts(har, split=int(0.6 * len(har)))
true_vol = g["vol"].shift(-1).loc[fc.index]              # the TRUE conditional volatility of tomorrow
pd.DataFrame({"correlation with the true volatility": {c: np.corrcoef(fc[c], true_vol)[0, 1] for c in ("naive", "har", "gbm")},
              "correlation with tomorrow's |return|": {c: np.corrcoef(fc[c], fc["target"])[0, 1] for c in ("naive", "har", "gbm")}}).round(3)

A linear HAR tracks the true GARCH volatility almost perfectly; gradient boosting on the same three inputs is much worse, because the noisy target (a single day's `|return|`) invites it to fit noise. Neither can predict tomorrow's `|return|` well: that is mostly luck. Judge volatility models against the latent truth when you have it, and against HAR always.

## 3. Rare events: warn, don't bet

`p.crash_warning` estimates the probability of a 4% fall within 5 days from recent volatility (a balanced logistic regression, trained only on resolved labels). Crashes happen on about 4% of days, so compare its **PR-AUC** with 4%, not with 0.5. Use the warning to **cut risk**: exposure 1, or `low` on the day *after* a warning (`prob.shift(1) > threshold`). Never short on it.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def risk_off_returns(ret, prob, threshold=0.5, low=0.5):
    expo = pd.Series(np.where(prob > threshold, low, 1.0), index=prob.index).shift(1).dropna()
    return ret.loc[expo.index] * expo

cw = p.crash_warning(g["ret"], split=1500)
cases = [(g["ret"], cw["prob"]), (g["ret"], cw["prob"], 0.7, 0.0)]
mine = [p.attempt(risk_off_returns, *c) for c in cases]
mine = p.check("risk_off_returns", mine, [p.risk_off_returns(*c) for c in cases])
hold = g["ret"].loc[mine[0].index]
print(f"crash base rate {cw['label'].mean():.3f}; PR-AUC {p.pr_auc(cw['prob'], cw['label']):.3f} "
      f"({p.pr_auc(cw['prob'], cw['label']) / cw['label'].mean():.1f}× the base rate); warnings on {(cw['prob'] > 0.5).mean():.0%} of days")
pd.DataFrame({"buy and hold": {"max drawdown": p.max_drawdown(hold), "Sharpe": p.sharpe(hold)},
              "halve exposure after a warning": {"max drawdown": p.max_drawdown(mine[0]), "Sharpe": p.sharpe(mine[0])}}).round(3)

About twice the base rate is a modest warning, but it is enough to shave the worst drawdown and even lift the Sharpe ratio a little, because crashes cluster in high-volatility periods. The same model used as a short signal would be wrong most of the time.

## Wrap-up

* Retrain on resolved labels only; walk forward; count every model family you try.
* Meta-labeling earns by sizing, not by predicting direction.
* Volatility: HAR first. Rare events: PR-AUC against the base rate, used for risk cuts.
* Graded version: `labs/part10/week36_strategies` (`walk_forward_meta`, `meta_summary`, `vol_forecasts`, `crash_warning`, `rank_ic`).